# 29 — Federated Learning Fundamentals — Lab

This lab simulates a federated learning system using the Flower framework. You will implement a FedAvg simulation on CIFAR-10 and observe how label skew (non-IID data distribution) degrades convergence compared to IID training. The lab reinforces the FedAvg algorithm, client-server orchestration, and the practical impact of data heterogeneity on federated learning.

- State the FedAvg algorithm step by step, including round structure, local training, weighted averaging, and convergence criteria.
- Distinguish cross-device federated learning from cross-silo federated learning by scale, trust model, and data quantity per client.
- Implement a FedAvg simulation using Flower with configurable numbers of virtual clients, local epochs, and participation fraction.
- Explain why non-IID label skew degrades FedAvg convergence and quantify the accuracy gap versus IID training on a fixed dataset.

You should have completed the m29 lecture and understand the FedAvg algorithm, the FedAvg pseudocode, cross-device and cross-silo FL settings, and the impact of non-IID data on convergence. Familiarity with training neural networks on CIFAR-10 and basic gradient descent hyperparameters is assumed.

Required software/packages: flower (flwr) >= 1.5.0, tensorflow >= 2.12.0, numpy >= 1.23.0, matplotlib >= 3.7.0, scipy.

Run the following cell to install and import required packages.

In [ ]:
import subprocess
import sys

# Install Flower if not present
try:
    import flwr
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "flwr>=1.5.0"])
    import flwr
    print("Flower installed successfully.")

In [ ]:


# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import flwr as fl
from scipy.stats import dirichlet

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = y_train.flatten(), y_test.flatten()
print(f"Setup complete. CIFAR-10 loaded: train shape {x_train.shape}, test shape {x_test.shape}")

Federated Learning trains a model by distributing training across multiple clients, each holding private data. The FedAvg algorithm orchestrates this: each round, the server broadcasts the global model to clients, each client trains locally for E epochs, clients upload updated weights, the server averages them and repeats. In this lab, you will simulate this flow using Flower on CIFAR-10. You will run two experiments: IID data (all clients see all classes equally) and non-IID data (Dirichlet label skew, clients see different subsets of classes). Comparing convergence curves will show how label skew degrades federated training.

1. Implement functions to create IID and non-IID partitions of CIFAR-10 across 10 clients, and implement a Flower NumPyClient with fit() and evaluate() methods. (6 min)
2. Run the IID simulation: partition CIFAR-10 training data uniformly across 10 virtual clients, run FedAvg for 20 rounds with C=1.0 (all clients participate), record global accuracy per round. (9 min)
3. Run the Non-IID simulation: re-partition the same CIFAR-10 dataset using Dirichlet alpha=0.1 label distribution, run the same 20-round FedAvg, record global accuracy per round. (9 min)
4. Plot both accuracy curves (IID and Non-IID) on the same axes, compute and report the final-round accuracy gap between IID and Non-IID runs. (6 min)

In [ ]:
# Starter code: Implement IID and non-IID partitioning functions with batch size parameter

def create_iid_partitions(x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray, num_clients: int) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Partition CIFAR-10 training and test data uniformly across clients (IID).
    Returns a list of (x_train_client, y_train_client, x_test_client, y_test_client) tuples.
    """
    # TODO: Implement IID partitioning
    # Hint: Split x_train and y_train uniformly across num_clients
    pass

def create_noniid_partitions(x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray, num_clients: int, alpha: float) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Partition CIFAR-10 data using Dirichlet label distribution (non-IID).
    Each client samples a label distribution from Dirichlet(alpha) for each class.
    Returns a list of (x_train_client, y_train_client, x_test_client, y_test_client) tuples.
    """
    # TODO: Implement non-IID partitioning using Dirichlet
    # Hint: For each client, sample label proportions from Dirichlet(alpha, ..., alpha) with 10 classes
    pass

print("Partitioning functions defined.")

In [ ]:
# Define model and Flower client

def create_model() -> keras.Model:
    """
    Create a simple CNN model for CIFAR-10.
    """
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

class FedAvgClient(fl.client.NumPyClient):
    """
    Flower NumPyClient for federated learning on CIFAR-10.
    """
    def __init__(self, x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray, y_test: np.ndarray):
        self.x_train = x_train
        self.y_train = y_train
        self.x_test = x_test
        self.y_test = y_test
        self.model = create_model()
    
    def get_parameters(self, config: Dict) -> List[np.ndarray]:
        return self.model.get_weights()
    
    def fit(self, parameters: List[np.ndarray], config: Dict) -> Tuple[List[np.ndarray], int, Dict]:
        """
        Set model parameters, train locally for E epochs with configurable batch size, return updated parameters.
        TODO: Implement local training with E=2 epochs from config and batch_size from config.
        """
        pass
    
    def evaluate(self, parameters: List[np.ndarray], config: Dict) -> Tuple[float, int, Dict]:
        """
        Set model parameters, evaluate on local test data, return loss and accuracy.
        TODO: Implement local evaluation.
        """
        pass

print("Model and Flower client defined.")

In [ ]:
# IID and Non-IID simulations with Flower
# TODO: Implement IID simulation
# 1. Call create_iid_partitions() to partition CIFAR-10 data across 10 clients
# 2. Define a function to create a Flower client for a given client index
# 3. Use fl.simulation.start_simulation() or fl.server.start_server() with FedAvg strategy
# 4. Run for 20 rounds with batch_size=32 and collect global accuracy per round

iid_accuracies = []  # List of 20 accuracy values, one per round
print("IID simulation: TODO")

# TODO: Implement non-IID simulation using the same approach with create_noniid_partitions(alpha=0.1) and batch_size=32
noniid_accuracies = []  # List of 20 accuracy values, one per round
print("Non-IID simulation: TODO")

In [ ]:
# Plot both accuracy curves and compute the accuracy gap
# TODO: Plot iid_accuracies and noniid_accuracies as two lines on the same axes
# TODO: Compute gap = iid_accuracies[-1] - noniid_accuracies[-1] and print

plt.figure(figsize=(10, 6))
# Plot code here
plt.xlabel('Round')
plt.ylabel('Global Accuracy')
plt.title('IID vs Non-IID Federated Learning Convergence')
# plt.legend()
# plt.show()

# Compute and print accuracy gap
# gap = ...
# print(f"Accuracy gap (IID - Non-IID): {gap:.4f}")
print("Expected output: Two accuracy curves, with IID accuracy higher than non-IID.")

Expected output: A plot showing two accuracy curves, one for IID (higher accuracy, smoother convergence) and one for non-IID (lower accuracy, more oscillation). The final-round IID accuracy should be approximately 0.65-0.75, and the non-IID accuracy approximately 0.50-0.60, resulting in a gap of 0.10-0.20.

Comprehension questions:

1. Why does the non-IID accuracy curve show more oscillation than the IID curve? What does this oscillation tell you about the aggregation process?
2. In your non-IID simulation, each client sees only a subset of CIFAR-10 classes due to Dirichlet(alpha=0.1) label skew. How does this affect the local training loss versus the global test accuracy?

In [ ]:
# Verify that both simulations completed successfully
assert isinstance(iid_accuracies, list) and len(iid_accuracies) == 20, "IID accuracies must be a list of 20 values."
assert isinstance(noniid_accuracies, list) and len(noniid_accuracies) == 20, "Non-IID accuracies must be a list of 20 values."
assert all(0 <= acc <= 1 for acc in iid_accuracies), "All IID accuracies must be between 0 and 1."
assert all(0 <= acc <= 1 for acc in noniid_accuracies), "All non-IID accuracies must be between 0 and 1."
assert iid_accuracies[-1] > noniid_accuracies[-1], "IID final accuracy must be higher than non-IID final accuracy."

print("PASS: All verification checks passed.")
print(f"IID final accuracy: {iid_accuracies[-1]:.4f}")
print(f"Non-IID final accuracy: {noniid_accuracies[-1]:.4f}")
print(f"Accuracy gap: {(iid_accuracies[-1] - noniid_accuracies[-1]):.4f}")

Optional: Extend the simulation by increasing local epochs E from 2 to 5 on the non-IID clients and re-run the simulation. Record whether the accuracy gap shrinks. This explores the communication-convergence trade-off: higher E reduces client drift but increases total training time.

No cleanup required. All data is held in memory in this notebook and will be cleared when the kernel is restarted.

Completion criteria:
- [ ] You have implemented the create_iid_partitions() function to uniformly partition CIFAR-10 across 10 clients.
- [ ] You have implemented the create_noniid_partitions() function using Dirichlet(alpha=0.1) label distribution.
- [ ] You have implemented the FedAvgClient.fit() method to run E=2 local SGD epochs with configurable batch_size and return updated weights.
- [ ] You have implemented the FedAvgClient.evaluate() method to test on local data and return loss and accuracy.
- [ ] You have run the IID simulation for 20 rounds with batch_size=32 and collected global accuracy values.
- [ ] You have run the non-IID simulation for 20 rounds with batch_size=32 and collected global accuracy values.
- [ ] You have plotted both accuracy curves on the same axes.
- [ ] You have computed and reported the accuracy gap between IID and non-IID final rounds.
- [ ] The verification cell prints PASS: all checks passed.
- [ ] You have answered the two comprehension questions.